# Subgraphs Demo

This notebook shows how to compile one LangGraph graph and use it as a node inside a larger parent graph.

In [ ]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END

In [ ]:
class ReportState(TypedDict, total=False):
    topic: str
    raw_notes: str
    cleaned_notes: str
    bullet_points: list[str]
    final_report: str

First, build a small subgraph that cleans notes and turns them into bullet points.

In [ ]:
def clean_notes(state: ReportState):
    cleaned = " ".join(state["raw_notes"].split())
    return {"cleaned_notes": cleaned}


def make_bullets(state: ReportState):
    sentences = [
        sentence.strip()
        for sentence in state["cleaned_notes"].split(".")
        if sentence.strip()
    ]

    bullets = [f"- {sentence}" for sentence in sentences]
    return {"bullet_points": bullets}

In [ ]:
notes_builder = StateGraph(ReportState)

notes_builder.add_node("clean_notes", clean_notes)
notes_builder.add_node("make_bullets", make_bullets)

notes_builder.add_edge(START, "clean_notes")
notes_builder.add_edge("clean_notes", "make_bullets")
notes_builder.add_edge("make_bullets", END)

notes_subgraph = notes_builder.compile()

notes_subgraph

Now use that compiled subgraph as one node in a parent graph.

In [ ]:
def write_report(state: ReportState):
    bullets = "\n".join(state["bullet_points"])
    report = f"Report: {state['topic']}\n\nKey points:\n{bullets}"
    return {"final_report": report}

In [ ]:
parent_builder = StateGraph(ReportState)

parent_builder.add_node("notes_subgraph", notes_subgraph)
parent_builder.add_node("write_report", write_report)

parent_builder.add_edge(START, "notes_subgraph")
parent_builder.add_edge("notes_subgraph", "write_report")
parent_builder.add_edge("write_report", END)

report_graph = parent_builder.compile()

report_graph

In [ ]:
result = report_graph.invoke(
    {
        "topic": "LangGraph subgraphs",
        "raw_notes": "Subgraphs keep logic modular. They can be reused as nodes. Parent graphs stay easier to read.",
    }
)

print(result["final_report"])

In [ ]:
# Text diagram of the parent graph.
print(report_graph.get_graph().draw_mermaid())